In [ ]:
import requests
import os
from dotenv import load_dotenv
import json
from datetime import datetime


In [ ]:
# Configurações do SonarQube
load_dotenv()
SONAR_TOKEN = os.getenv("SONAR_TOKEN", "" )
BASE_URL_SONAR = os.getenv("SONAR_HOST", "" )

In [20]:
# Cabeçalhos para autenticação
headers = {
    "Authorization": f"Bearer {SONAR_TOKEN}"
}

In [ ]:

# Diretório de saída
output_dir = "analytics-raw-data"
os.makedirs(output_dir, exist_ok=True)

# Métricas desejadas
METRICS_SONAR = [
    "files", "functions", "complexity", "comment_lines_density", "duplicated_lines_density",
    "coverage", "ncloc", "tests", "test_errors", "test_failures", "test_execution_time",
    "security_rating", "blocker_violations", "critical_violations", "bugs"
]

# Base URL do SonarQube
headers = {"Authorization": "Bearer {SONAR_TOKEN}"}  # Substituir pelo token correto

# Obter a lista de projetos no SonarQube
response_projects = requests.get(f"{BASE_URL_SONAR}/api/projects/search", headers=headers)
seconds=1
if response_projects.status_code == 200:
    projects_data = response_projects.json()
    projects = [p["key"] for p in projects_data.get("components", [])]

    for project_key in projects:  # Loop para cada projeto
        print(f"🔍 Analisando projeto: {project_key}")

        # Obter métricas detalhadas do SonarQube
        response_metrics = requests.get(
            f"{BASE_URL_SONAR}/api/measures/component_tree?component={project_key}&metricKeys={','.join(METRICS_SONAR)}",
            headers=headers
        )

        if response_metrics.status_code == 200:
            metrics_data = response_metrics.json()

            # Criar um nome de arquivo único com data, hora e formato de project_key adequado
            timestamp = datetime.now().strftime("%m-%d-%Y-%H-%M")
            key_suffix = project_key.split("-")[-1][-4:]
            seconds +=1
            if seconds >= 60:
                seconds = 0 

            filename = f"{project_key}.json"
            file_path = os.path.join(output_dir, filename)

            # Salvar o JSON das métricas
            with open(file_path, "w", encoding="utf-8") as json_file:
                json.dump(metrics_data, json_file, indent=4, ensure_ascii=False)

            print(f"✅ JSON criado para o projeto: {file_path}")
        else:
            print(f"⚠️ Erro ao buscar métricas para {project_key}, código {response_metrics.status_code}")

else:
    print("⚠️ Erro ao buscar a lista de projetos do SonarQube")